In [ ]:
import os
import xarray as xr
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# CONFIGURATION
# =========================================================
DATA_DIR = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/final_modified_sequence_with_yield"
YEARS = range(1991, 2024) # All available data

# Define the variables to check
# We will split them into "Raw Climate" and "Drought Indices" for cleaner plots
VARS_RAW = ['monthly_rain', 'max_temp', 'min_temp', 'radiation', 'vp', 'vp_deficit']
VARS_INDICES = ['spi_1', 'spi_3', 'spi_6', 'spi_12', 'spei_1', 'spei_3', 'spei_6', 'spei_12']

def run_experiment_1():
    print("📊 Starting Experiment 1: Data Diagnostics...")

    # Storage for flattened data
    data_storage = {var: [] for var in VARS_RAW + VARS_INDICES}
    data_storage['yield'] = []

    # 1. Load Data (Sub-sample if RAM is tight, but full data is better)
    # We will load every 10th pixel to speed up calculation without losing statistical significance
    stride = 10

    for year in YEARS:
        try:
            fpath = os.path.join(DATA_DIR, f"final_dataset_{year}.nc")
            ds = xr.open_dataset(fpath)

            # Get Yield Mask (Valid Farms)
            y_map = ds['yield'].values
            mask = ~np.isnan(y_map)

            # Store Yield
            data_storage['yield'].append(y_map[mask][::stride])

            # Store Climate Vars (Avg over season? Or correlate per month?)
            # CHALLENGE: You have 6 months (May-Oct).
            # A simple correlation matrix usually needs 1 value per year.
            # OPTION A: Correlate "September Rain" vs Yield.
            # OPTION B: Correlate "Seasonal Average Rain" vs Yield.

            # Let's do OPTION A: We will create columns like "Rain_May", "Rain_Jun", etc.
            # WAIT: That would be 14 vars * 6 months = 84 rows in heatmap. Too big.

            # BETTER APPROACH:
            # We will calculate correlation for "Critical Window" (e.g. Sep/Oct) vs "Early Season" (May/Jun).
            # But for simplicity, let's first check correlation of the VARIABLE (averaged over growing season) vs Yield.

            for var in VARS_RAW + VARS_INDICES:
                # Shape: (6, H, W) -> Mean over time -> (H, W)
                # We interpret this as "Seasonal Average Condition"
                val_map = ds[var].mean(dim='time').values

                # Check for NaNs in inputs (Ocean)
                val_map = np.nan_to_num(val_map, nan=0.0)

                data_storage[var].append(val_map[mask][::stride])

        except FileNotFoundError:
            print(f"Skipping {year}")

    # 2. Convert to DataFrame
    print("   Constructing DataFrame...")
    # Concatenate all years
    df_dict = {}
    for key in data_storage:
        if len(data_storage[key]) > 0:
            df_dict[key] = np.concatenate(data_storage[key])

    df = pd.DataFrame(df_dict)

    # 3. Calculate Correlation Matrix
    print("   Calculating Correlations...")
    # We only care about correlation with 'yield'
    corr_matrix = df.corr()[['yield']].sort_values(by='yield', ascending=False)

    # Drop the yield row itself
    corr_matrix = corr_matrix.drop('yield')

    # 4. Plotting
    plt.figure(figsize=(8, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
    plt.title("Correlation with Wheat Yield\n(Seasonal Average 1991-2023)")
    plt.ylabel("Climate Variable")
    plt.show()

    return df

# Run it
df_diagnostic = run_experiment_1()

Train loader and Test loader

In [ ]:
import os
import torch
import xarray as xr
import numpy as np
from torch.utils.data import Dataset, DataLoader

class WheatYieldDataset(Dataset):
    def __init__(self, data_dir, years, mode='train', scaler=None):
        """
        Args:
            data_dir (str): Path to 'final_sequences_with_yield'
            years (list): List of years for this split (e.g. 1991-2010)
            mode (str): 'train', 'val', or 'test'
            scaler (dict): Mean/Std dict from training set (needed for val/test)
        """
        self.files = [os.path.join(data_dir, f"final_dataset_{y}.nc") for y in years]
        self.mode = mode

        # 1. Define Variables (The Order Matters!)
        # Ensure these match your NetCDF variable names exactly
        self.feature_vars = [
            'monthly_rain', 'max_temp', 'min_temp', 'radiation',
            'vp', 'vp_deficit',
            'spi_1', 'spi_3', 'spi_6', 'spi_12',
            'spei_1', 'spei_3', 'spei_6', 'spei_12'
        ]

        # 2. Load Data into RAM
        # Since we have ~33 files (approx 500MB total), we can load it all into RAM
        # for extremely fast training speed.
        self.X_data = []
        self.y_data = []

        print(f"[{mode.upper()}] Loading {len(years)} years...")

        for fpath in self.files:
            try:
                ds = xr.open_dataset(fpath)

                # A. Create Yield Mask (Where yield is NOT NaN)
                y_np = ds['yield'].values # Shape: (lat, lon)
                mask = ~np.isnan(y_np)    # Boolean mask of valid farms

                # B. Extract Inputs (Features)
                year_features = []
                for var in self.feature_vars:
                    # Shape: (Time=6, Lat, Lon)
                    data = ds[var].values

                    # Select only valid pixels -> Result: (Time=6, N_valid_pixels)
                    valid_pixels = data[:, mask]
                    year_features.append(valid_pixels)

                # Stack features
                # Current shape: List of (6, N).
                # Stack to (Channels, 6, N) then Transpose to (N, 6, Channels)
                X_year = np.stack(year_features, axis=0)
                X_year = np.transpose(X_year, (2, 1, 0))

                # C. Extract Targets
                y_year = y_np[mask] # Shape: (N,)

                # D. Handle NaNs in INPUTS (e.g., if SPI is missing but Yield exists)
                # We replace input NaNs with 0.0 (which is the mean in Z-score world)
                X_year = np.nan_to_num(X_year, nan=0.0)

                self.X_data.append(X_year)
                self.y_data.append(y_year)

            except FileNotFoundError:
                print(f"⚠️ Missing file: {fpath}")

        # Concatenate all years into one giant Tensor
        if len(self.X_data) > 0:
            self.X_data = np.concatenate(self.X_data, axis=0) # (Total_Samples, 6, 14)
            self.y_data = np.concatenate(self.y_data, axis=0) # (Total_Samples,)
            print(f"[{mode.upper()}] Loaded {self.X_data.shape[0]} samples.")
        else:
            print(f"[{mode.upper()}] ❌ No data loaded!")
            return

        # 3. Normalization (Standardization)
        # We MUST calculate mean/std only on TRAIN data, then apply to Val/Test
        if mode == 'train':
            print("   Computing statistics...")
            # Calculate mean/std across Samples (0) and Time (1)
            mean = np.mean(self.X_data, axis=(0, 1))
            std = np.std(self.X_data, axis=(0, 1))
            self.scaler = {'mean': mean, 'std': std}
        else:
            self.scaler = scaler

        # Apply (X - Mean) / Std
        # Add 1e-6 to std to avoid division by zero
        self.X_data = (self.X_data - self.scaler['mean']) / (self.scaler['std'] + 1e-6)

        # Convert to PyTorch Tensors
        self.X_data = torch.tensor(self.X_data, dtype=torch.float32)
        self.y_data = torch.tensor(self.y_data, dtype=torch.float32).unsqueeze(1) # (N, 1)

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):
        return self.X_data[idx], self.y_data[idx]

# =========================================================
# HELPER: Get All Loaders
# =========================================================
def get_dataloaders(data_dir, batch_size=1024):
    # Standard Time-Series Split
    train_years = range(1991, 2011) # 20 Years
    val_years   = range(2011, 2018) # 7 Years
    test_years  = range(2018, 2024) # 6 Years (The "Future")

    # 1. Train
    train_ds = WheatYieldDataset(data_dir, train_years, mode='train')

    # 2. Val (Pass the train scaler!)
    val_ds = WheatYieldDataset(data_dir, val_years, mode='val', scaler=train_ds.scaler)

    # 3. Test (Pass the train scaler!)
    test_ds = WheatYieldDataset(data_dir, test_years, mode='test', scaler=train_ds.scaler)

    # Create PyTorch DataLoaders
    # num_workers=2 speeds up data fetching
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [ ]:
# Update this path to your "final_sequences_with_yield" folder
DATA_DIR = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/final_modified_sequence_with_yield"

# Initialize Loaders
train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, batch_size=1024)

# Grab one batch to check shapes
X_batch, y_batch = next(iter(train_loader))

print(f"\n✅ Data Loading Successful!")
print(f"Input Batch Shape:  {X_batch.shape}")
print(f"Target Batch Shape: {y_batch.shape}")

# EXPECTED OUTPUT:
# Input:  torch.Size([1024, 6, 14])  -> (Batch, Time, Variables)
# Target: torch.Size([1024, 1])      -> (Batch, Yield)

In [ ]:
import torch
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib

# Re-use your existing 'get_dataloaders' from the previous setup
# to ensure we use the EXACT same Train/Test split.

def run_experiment_2(train_loader, test_loader):
    print("🌲 Starting Experiment 2: Random Forest Baseline...")

    # 1. Flatten the Time-Series Data
    # LSTM Input: (Batch, Time=6, Feat=14)
    # RF Input:   (Batch, Time*Feat=84)

    def flatten_loader(loader):
        X_list, y_list = [], []
        for X, y in loader:
            # Flatten (Batch, 6, 14) -> (Batch, 84)
            batch_flat = X.reshape(X.shape[0], -1).numpy()
            X_list.append(batch_flat)
            y_list.append(y.numpy().ravel())

        return np.concatenate(X_list), np.concatenate(y_list)

    print("   Flattening data...")
    X_train, y_train = flatten_loader(train_loader)
    X_test, y_test = flatten_loader(test_loader)

    print(f"   Train Shape: {X_train.shape}")
    print(f"   Test Shape:  {X_test.shape}")

    # 2. Train Random Forest
    # n_jobs=-1 uses all CPU cores
    rf = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)

    print("   Training (this may take 5-10 mins)...")
    rf.fit(X_train, y_train)

    # 3. Evaluate
    print("   Predicting...")
    y_pred = rf.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n📊 RANDOM FOREST RESULTS:")
    print(f"-------------------------")
    print(f"RMSE: {rmse:.4f} t/ha")
    print(f"R²:   {r2:.4f}")

    # 4. Save Model & Plot
    joblib.dump(rf, "rf_baseline.pkl")

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.1, s=1, color='gray')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel("Observed Yield")
    plt.ylabel("RF Predicted Yield")
    plt.title(f"Static Baseline (Random Forest)\nR² = {r2:.3f}")
    plt.show()

# --- HOW TO RUN ---
# Assuming you have 'train_loader' and 'test_loader' from your main data script:
run_experiment_2(train_loader, test_loader)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
import time

# =========================================================
# MODEL DEFINITION: STANDARD LSTM
# =========================================================
class StandardLSTM(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=128, n_layers=2, dropout=0.2):
        super(StandardLSTM, self).__init__()

        # LSTM Layer: Reads the sequence
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        # Head: Predicts from the FINAL hidden state
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # x: (Batch, Time=6, Vars=14)
        lstm_out, _ = self.lstm(x)

        # We take only the LAST time step (October's state)
        # This implies the model must "remember" May-Sept in this final vector
        last_hidden = lstm_out[:, -1, :]

        return self.fc(last_hidden)

# =========================================================
# TRAINING FUNCTION
# =========================================================
def run_experiment_3(train_loader, val_loader, test_loader, device=None):
    print("⏳ Starting Experiment 3: Standard LSTM...")

    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"   Using device: {device}")

    model = StandardLSTM().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    # Training Loop
    epochs = 30
    best_loss = float('inf')
    train_losses, val_losses = [], []

    print(f"   Training for {epochs} epochs...")
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        batch_losses = []
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            preds = model(X)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = np.mean(batch_losses)
        train_losses.append(train_loss)

        # Validation
        model.eval()
        val_batch_losses = []
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                preds = model(X)
                loss = criterion(preds, y)
                val_batch_losses.append(loss.item())

        val_loss = np.mean(val_batch_losses)
        val_losses.append(val_loss)

        # Save Best
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), "best_lstm_standard.pth")

        if (epoch+1) % 5 == 0:
            print(f"   Epoch {epoch+1}/{epochs} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")

    print(f"   Training Time: {(time.time()-start_time)/60:.1f} mins")

    # =========================================================
    # EVALUATION
    # =========================================================
    print("   Evaluating on Test Set (2018-2023)...")
    model.load_state_dict(torch.load("best_lstm_standard.pth"))
    model.eval()

    all_preds, all_true = [], []
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            preds = model(X)
            all_preds.append(preds.cpu().numpy())
            all_true.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_true)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n📊 STANDARD LSTM RESULTS:")
    print(f"-------------------------")
    print(f"RMSE: {rmse:.4f} t/ha")
    print(f"R²:   {r2:.4f}")

    # Plot
    plt.figure(figsize=(6,6))
    plt.scatter(y_true, y_pred, alpha=0.1, s=1, color='blue')
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')
    plt.title(f"Temporal Baseline (Standard LSTM)\nR² = {r2:.3f}")
    plt.xlabel("Observed Yield")
    plt.ylabel("LSTM Predicted Yield")
    plt.show()

# Run it
run_experiment_3(train_loader, val_loader, test_loader)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# =========================================================
# 1. THE ARCHITECTURE: Residual LSTM + Attention
# =========================================================
class ResidualLSTM(nn.Module):
    """
    A single LSTM layer with a Residual Skip Connection and LayerNorm.
    """
    def __init__(self, input_dim, hidden_dim, dropout=0.2):
        super(ResidualLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)

        # Project input if dimensions mismatch (e.g. 14 -> 64)
        if input_dim != hidden_dim:
            self.project = nn.Linear(input_dim, hidden_dim)
        else:
            self.project = None

    def forward(self, x):
        # x: (Batch, Time, Input_Dim)
        out, _ = self.lstm(x) # (Batch, Time, Hidden)
        out = self.dropout(out)

        # Residual Connection
        residual = self.project(x) if self.project else x

        # Add & Normalize
        return self.layer_norm(out + residual)

class DeepYieldResNet(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=128, n_layers=2, dropout=0.2):
        super(DeepYieldResNet, self).__init__()

        # Stack of Residual LSTMs
        self.layers = nn.ModuleList()
        # Layer 1: Input -> Hidden
        self.layers.append(ResidualLSTM(input_dim, hidden_dim, dropout))
        # Layer 2+: Hidden -> Hidden
        for _ in range(n_layers - 1):
            self.layers.append(ResidualLSTM(hidden_dim, hidden_dim, dropout))

        # Attention Mechanism
        self.att_fc = nn.Linear(hidden_dim, 1, bias=False)

        # Regression Head
        self.fc_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (Batch, 6, 14)

        # Pass through LSTM Stack
        curr_out = x
        for layer in self.layers:
            curr_out = layer(curr_out)

        # --- Attention ---
        # energy: (Batch, 6, 1)
        energy = torch.tanh(self.att_fc(curr_out))
        # weights: (Batch, 6, 1) -> sum to 1 across time
        weights = F.softmax(energy, dim=1)

        # Context Vector: Weighted sum of all time steps
        context = torch.sum(weights * curr_out, dim=1)

        # Prediction
        prediction = self.fc_head(context)

        return prediction, weights

# =========================================================
# 2. TRAINING FUNCTION (Same as before, adapted for tuple output)
# =========================================================
def run_experiment_4(train_loader, val_loader, test_loader, device=None):
    print("🚀 Starting Experiment 4: Proposed Res-Attn-LSTM...")

    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"   Using device: {device}")

    model = DeepYieldResNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    epochs = 40 # Give it a bit more time to converge
    best_loss = float('inf')

    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        batch_losses = []
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()

            # Model returns (pred, weights)
            preds, _ = model(X)

            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = np.mean(batch_losses)

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                preds, _ = model(X)
                loss = criterion(preds, y)
                val_losses.append(loss.item())
        val_loss = np.mean(val_losses)

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), "best_res_attn_model.pth")

        if (epoch+1) % 5 == 0:
            print(f"   Epoch {epoch+1}/{epochs} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")

    print(f"   Training Time: {(time.time()-start_time)/60:.1f} mins")

    # Evaluation
    print("   Evaluating Proposed Model...")
    model.load_state_dict(torch.load("best_res_attn_model.pth"))
    model.eval()

    all_preds, all_true = [], []
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            preds, _ = model(X)
            all_preds.append(preds.cpu().numpy())
            all_true.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_true)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n🏆 PROPOSED MODEL RESULTS:")
    print(f"-------------------------")
    print(f"RMSE: {rmse:.4f} t/ha")
    print(f"R²:   {r2:.4f}")

    # Save for next steps (Attention Analysis)
    return model

# Run it
final_model = run_experiment_4(train_loader, val_loader, test_loader)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time

# Define the Middle Model (Attention, but NO Residuals)
class LSTM_Attention_Simple(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=128, n_layers=2, dropout=0.2):
        super(LSTM_Attention_Simple, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )
        self.att_fc = nn.Linear(hidden_dim, 1, bias=False)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x) # (Batch, 6, Hidden)

        # Attention
        energy = torch.tanh(self.att_fc(out))
        weights = F.softmax(energy, dim=1)
        context = torch.sum(weights * out, dim=1)

        return self.fc(context), weights

# Training Function that RETURNS the history (for plotting)
def train_and_track(model, name, train_loader, val_loader, test_loader, device=None):
    print(f"🚀 Training {name}...")

    # Determine device dynamically
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"   Using device: {device}")

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    history = {'train': [], 'val': []}
    best_loss = float('inf')

    for epoch in range(30): # 30 Epochs is enough since it overfits early
        model.train()
        losses = []
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()

            # Handle models with different returns (tuple vs tensor)
            out = model(X)
            if isinstance(out, tuple): preds = out[0]
            else: preds = out

            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        avg_train = np.mean(losses)

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                if isinstance(out, tuple): preds = out[0]
                else: preds = out
                val_losses.append(criterion(preds, y).item())
        avg_val = np.mean(val_losses)

        history['train'].append(avg_train)
        history['val'].append(avg_val)

        # Save Best
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), f"best_{name}.pth")

        if (epoch+1) % 5 == 0:
            print(f"   Epoch {epoch+1} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")

    # Load Best for Test
    model.load_state_dict(torch.load(f"best_{name}.pth"))
    model.eval()

    # Test Evaluation
    preds_list, true_list = [], []
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            if isinstance(out, tuple): p = out[0]
            else: p = out
            preds_list.append(p.cpu().numpy())
            true_list.append(y.cpu().numpy())

    y_p = np.concatenate(preds_list)
    y_t = np.concatenate(true_list)
    rmse = np.sqrt(np.mean((y_t - y_p)**2))
    r2 = r2_score(y_t, y_p)

    print(f"📊 {name} RESULT -> RMSE: {rmse:.4f} | R²: {r2:.4f}")
    return history, model

# --- RUN IT ---
# 1. Initialize Model
middle_model = LSTM_Attention_Simple()

# 2. Train
history_attn, final_middle_model = train_and_track(middle_model, "LSTM_Attn_NoRes", train_loader, val_loader, test_loader)


In [ ]:
import matplotlib.pyplot as plt

def plot_loss_comparison(histories):
    plt.figure(figsize=(12, 5))

    colors = {'LSTM_Attn_NoRes': 'blue', 'Proposed_Res_Attn': 'green'}

    for name, hist in histories.items():
        # Plot Validation Loss (Solid Line)
        plt.plot(hist['val'], label=f"{name} (Val)", color=colors.get(name, 'gray'), linewidth=2)
        # Plot Training Loss (Dashed Line)
        plt.plot(hist['train'], label=f"{name} (Train)", color=colors.get(name, 'gray'), linestyle='--', alpha=0.6)

    plt.xlabel("Epochs")
    plt.ylabel("MSE Loss")
    plt.title("Training Dynamics: Does Residual Connection Stabilize Learning?")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 0.4) # Focus on the lower range
    plt.show()

# You need to have the history from the Proposed Model too.
# If you didn't save 'history' from Exp 4, you might need to re-run Exp 4 quickly to capture the list.
# OR, just plot the 'history_attn' we just generated to verify it works.
plot_loss_comparison({'LSTM_Attn_NoRes': history_attn})

In [ ]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Re-Initialize the Architecture (Must match the 0.59 model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
champion_model = DeepYieldResNet(input_dim=14, hidden_dim=128, n_layers=2).to(device)

# 2. Load the Best Weights (from Experiment 4)
try:
    champion_model.load_state_dict(torch.load("best_res_attn_model.pth", map_location=device))
    print("✅ Successfully loaded 'best_res_attn_model.pth' (The 0.59 R² Model)")
except FileNotFoundError:
    print("❌ Error: Could not find 'best_res_attn_model.pth'. Did you run Experiment 4?")

# 3. Run Experiment 5: Attention Analysis
def run_experiment_5_final(model, test_loader):
    print("🧠 Starting Experiment 5: Extracting Phenological Signals...")
    model.eval()

    all_weights = []

    with torch.no_grad():
        for X, _ in test_loader:
            X = X.to(device)
            # Model returns (preds, weights)
            _, weights = model(X)
            # Squeeze: (Batch, 6, 1) -> (Batch, 6)
            all_weights.append(weights.squeeze(2).cpu().numpy())

    # Concatenate
    attention_matrix = np.concatenate(all_weights, axis=0)

    # Plotting
    months = ['May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct']
    df_att = pd.DataFrame(attention_matrix, columns=months)

    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df_att, palette="viridis")
    plt.title("Phenological Sensitivity: Which Months Drive the Forecast?\n(Extracted from DeepYield-ResNet)")
    plt.ylabel("Attention Weight (Importance)")
    plt.xlabel("Month")
    plt.grid(True, alpha=0.3)
    plt.show()

    print("   Mean Importance per Month:")
    print(df_att.mean())

# Run on Test Set
run_experiment_5_final(champion_model, test_loader)

In [ ]:
def run_experiment_6_final(model, test_loader):
    print("🔬 Starting Experiment 6: Driver Ablation Analysis...")
    model.eval()
    criterion = torch.nn.MSELoss()

    # 1. Get Baseline MSE
    all_X, all_y = [], []
    for X, y in test_loader:
        all_X.append(X)
        all_y.append(y)
    X_full = torch.cat(all_X).to(device)
    y_full = torch.cat(all_y).to(device)

    with torch.no_grad():
        base_pred, _ = model(X_full)
        base_mse = criterion(base_pred, y_full).item()

    # 2. Permute Groups
    groups = {
        'Raw Climate (Rain/Temp)': [0,1,2,3,4,5],
        'Precipitation Memory (SPI)': [6,7,8,9],
        'Evaporative Memory (SPEI)': [10,11,12,13]
    }

    results = []

    for group_name, indices in groups.items():
        X_perm = X_full.clone()

        # Shuffle these columns
        for idx in indices:
            perm_idx = torch.randperm(X_perm.size(0))
            X_perm[:, :, idx] = X_perm[perm_idx, :, idx]

        with torch.no_grad():
            pred_p, _ = model(X_perm)
            mse_p = criterion(pred_p, y_full).item()

        rmse_increase = np.sqrt(mse_p) - np.sqrt(base_mse)
        results.append({'Driver Group': group_name, 'RMSE Increase': rmse_increase})
        print(f"   {group_name}: +{rmse_increase:.4f} t/ha Error")

    # Plot
    import pandas as pd
    df = pd.DataFrame(results)
    plt.figure(figsize=(8, 5))
    sns.barplot(data=df, x='Driver Group', y='RMSE Increase', palette='magma')
    plt.title("Ablation Study: What Information is Essential?")
    plt.ylabel("Loss in Accuracy (Increase in RMSE)")
    plt.show()

run_experiment_6_final(champion_model, test_loader)

In [ ]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def run_experiment_6_drivers(model, test_loader, device=None):
    print("🔬 Starting Experiment 6: Driver Importance Analysis...")
    # Determine device dynamically if not provided
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"   Using device: {device}")

    model.eval()
    criterion = nn.MSELoss()

    # 1. Get Baseline MSE
    all_X, all_y = [], []
    for X, y in test_loader:
        all_X.append(X)
        all_y.append(y)
    X_full = torch.cat(all_X).to(device)
    y_full = torch.cat(all_y).to(device)

    with torch.no_grad():
        base_pred, _ = model(X_full)
        base_mse = criterion(base_pred, y_full).item()

    # 2. Define Groups
    # Adjust these indices if your variable order is different!
    # Based on your Dataset class:
    # 0-5: Raw (Rain, MaxT, MinT, Rad, VP, VPD)
    # 6-9: SPI (1, 3, 6, 12)
    # 10-13: SPEI (1, 3, 6, 12)
    groups = {
        'Raw Climate (Rain/Temp)': [0,1,2,3,4,5],
        'Precipitation Memory (SPI)': [6,7,8,9],
        'Evaporative Memory (SPEI)': [10,11,12,13]
    }

    results = []

    for group_name, indices in groups.items():
        X_perm = X_full.clone()

        # Shuffle columns in this group
        for idx in indices:
            perm_idx = torch.randperm(X_perm.size(0))
            X_perm[:, :, idx] = X_perm[perm_idx, :, idx]

        with torch.no_grad():
            pred_p, _ = model(X_perm)
            mse_p = criterion(pred_p, y_full).item()

        rmse_increase = np.sqrt(mse_p) - np.sqrt(base_mse)
        results.append({'Driver Group': group_name, 'RMSE Increase': rmse_increase})
        print(f"   {group_name}: +{rmse_increase:.4f} t/ha Error")

    # Plot
    df = pd.DataFrame(results)
    plt.figure(figsize=(8, 5))
    sns.barplot(data=df, x='Driver Group', y='RMSE Increase', palette='magma')
    plt.title("Ablation Study: What Information is Essential?")
    plt.ylabel("Loss in Accuracy (Increase in RMSE)")
    plt.show()

# Run it
run_experiment_6_drivers(final_model, test_loader)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import torch

def run_experiment_7_spatial_map(model, data_dir, year, scaler, device=None):
    print(f"🌍 Starting Experiment 7: Spatial Error Mapping for {year}...")

    # Determine device dynamically if not provided
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"   Using device: {device}")

    # 1. Load the single file manually to keep spatial structure
    fpath = os.path.join(data_dir, f"final_dataset_{year}.nc")
    try:
        ds = xr.open_dataset(fpath)
    except FileNotFoundError:
        print(f"❌ File not found: {fpath}")
        return

    # 2. Extract Features exactly like the Dataset Class
    # We need to stack them: (Time, Lat, Lon)
    feature_vars = [
        'monthly_rain', 'max_temp', 'min_temp', 'radiation',
        'vp', 'vp_deficit',
        'spi_1', 'spi_3', 'spi_6', 'spi_12',
        'spei_1', 'spei_3', 'spei_6', 'spei_12'
    ]

    # Get Yield Mask (Where yield exists)
    y_true_map = ds['yield'].values
    mask = ~np.isnan(y_true_map)

    # Prepare Input Array
    # Shape: (14, 6, H, W) -> but we need flattened pixels for the model
    # Let's extract valid pixels first
    features_list = []
    for var in feature_vars:
        data = ds[var].values # (6, H, W)
        valid_pixels = data[:, mask] # (6, N_pixels)
        features_list.append(valid_pixels)

    # Stack: (14, 6, N) -> Transpose to (N, 6, 14)
    X_spatial = np.stack(features_list, axis=0)
    X_spatial = np.transpose(X_spatial, (2, 1, 0))

    # Handle NaNs (Ocean)
    X_spatial = np.nan_to_num(X_spatial, nan=0.0)

    # 3. Normalize (Using training scaler)
    # We must replicate the scaler logic manually here
    # (X - mean) / std
    # We assume 'scaler' is the dict passed in
    mean = scaler['mean']
    std = scaler['std']

    X_spatial = (X_spatial - mean) / (std + 1e-6)

    # 4. Predict in Chunks (to avoid Memory Error)
    model.eval()
    X_tensor = torch.tensor(X_spatial, dtype=torch.float32)

    batch_size = 4096
    all_preds = []

    with torch.no_grad():
        for i in range(0, len(X_tensor), batch_size):
            X_batch = X_tensor[i:i+batch_size].to(device)
            # Model returns (preds, weights). We only want preds.
            preds, _ = model(X_batch)
            all_preds.append(preds.cpu().numpy())

    # Concatenate predictions
    y_pred_flat = np.concatenate(all_preds, axis=0).flatten()

    # 5. Reconstruct the Map
    pred_map = np.full(y_true_map.shape, np.nan)
    pred_map[mask] = y_pred_flat

    # Calculate Error (Observed - Predicted)
    # Positive = Model Underestimated
    # Negative = Model Overestimated
    error_map = y_true_map - pred_map

    # 6. Plotting
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Observed
    im1 = axes[0].imshow(y_true_map, cmap='viridis', vmin=0, vmax=4)
    axes[0].set_title(f"Observed Yield {year} (t/ha)")
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

    # Predicted
    im2 = axes[1].imshow(pred_map, cmap='viridis', vmin=0, vmax=4)
    axes[1].set_title(f"Predicted Yield {year} (t/ha)\n(DeepYield-ResNet)")
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

    # Error
    # Use 'RdBu' (Red-Blue) centered at 0
    im3 = axes[2].imshow(error_map, cmap='RdBu', vmin=-1.5, vmax=1.5)
    axes[2].set_title(f"Residual Map (Observed - Predicted)\nBlue = Overestimation | Red = Underestimation")
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

# --- RUN IT ---
# We need the 'scaler' from the training step.
# Since we didn't save it explicitly in the last run, we can quickly re-compute it
# from the train_loader dataset.
DATA_DIR = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/final_modified_sequence_with_yield"

# Temporarily init a dataset just to get mean/std
temp_ds = WheatYieldDataset(DATA_DIR, range(1991, 2011), mode='train')
scaler = temp_ds.scaler

# Run for 2019 (A spatially interesting drought year)
run_experiment_7_spatial_map(champion_model, DATA_DIR, 2019, scaler)
